# Digit Recognizer — 동결 분류기 + KNN 게이팅으로 신규 클래스 추가 (Colab)

**연습 기법** (IOAI 2024 *Help BOBAI* 와 동일): 이미 **배포된 분류기를 재학습하지 않고**, 새 클래스의
라벨 데이터를 **거리(KNN)** 로만 활용해 분류 가능 클래스를 늘린다.

**시나리오 (BOBAI 1:1 매핑)**:
- 숫자 **0~4** 는 *이미 배포된* 5-way 분류기가 담당 → **파라미터 변경 금지(동결)**.
- 숫자 **5~9** 는 나중에 추가된 신규 클래스. 라벨 데이터는 있지만 배포 모델을 다시 학습할 수 없어,
  **KNN(비모수적 거리법)** 으로 '신규 클래스인지' 게이팅한 뒤 해당 숫자를 배정.
- 게이트가 '기존(0)' 이라고 하면 → 동결 5-way 분류기 사용. (BOBAI 의 `which+4` 로직 일반화)

**평가**: 캐글 리더보드는 정확도, 여기서는 BOBAI 처럼 **macro-F1 · per-class F1** 도 함께 본다.
코드는 `scikit-learn` 기본 함수만 사용(LogisticRegression + KNeighborsClassifier).


## 0. 라이브러리 설치 & Kaggle 자격증명


In [ ]:
import sys, subprocess
for pkg in ["kaggle", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")


In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")


## 1. Kaggle 에서 Digit Recognizer 데이터 다운로드


In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("digit-recognizer", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])


## 2. 데이터 준비 — 기존(0~4) vs 신규(5~9) 시나리오
`train.csv` 로 학습/검증을 만들고, `test.csv` 는 캐글 제출용 예측에 사용한다.


In [ ]:
import os, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

train = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))
y = train["label"].to_numpy()
X = train.drop(columns=["label"]).to_numpy(dtype=np.float32) / 255.0
X_test = test.to_numpy(dtype=np.float32) / 255.0

# 학습/검증 분할 (검증으로 기법 점검)
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("train:", X_tr.shape, " val:", X_val.shape, " test:", X_test.shape)
print("기존 클래스 0~4, 신규 클래스 5~9")


## 3. 배포된 5-way 분류기 학습 후 **동결** (기존 0~4 만)
실제 배포 상황을 모사: 0~4 로만 분류기를 만들고, 이후 파라미터를 바꾸지 않는다.


In [ ]:
from sklearn.linear_model import LogisticRegression

mask_old = y_tr < 5                       # 기존 클래스 데이터만
base_clf = LogisticRegression(max_iter=200, n_jobs=-1)
base_clf.fit(X_tr[mask_old], y_tr[mask_old])
print("동결 5-way 분류기 학습 완료 (클래스:", sorted(set(y_tr[mask_old])), ")")
# 이후 base_clf 는 재학습/수정하지 않는다 (BOBAI 제약).


## 4. 신규 클래스(5~9)를 **KNN 게이팅**으로 추가 — 재학습 없이, 거리만 사용
라벨을 `{기존<5 → 0, 5 → 1, …, 9 → 5}` 로 재라벨해 KNN 이 신규 클래스를 가려낸다.
(BOBAI: `{기존→0, 5→1, 6→2}` 의 일반화)


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# 게이트 라벨: 0=기존(0~4), 1~5 = 신규 숫자 5~9
gate_labels = np.where(y_tr < 5, 0, y_tr - 4)

# KNN 참조셋은 BOBAI 규모(수천)로 서브샘플 → 가볍고 빠르게
rng = np.random.RandomState(0)
idx = rng.permutation(len(X_tr))[:12000]
knn = KNeighborsClassifier(n_neighbors=3, weights="distance")
knn.fit(X_tr[idx], gate_labels[idx])
print("KNN 게이트 학습 완료 — 게이트 분포:", np.bincount(gate_labels[idx]))

def predict_incremental(Xq):
    """동결 5-way + KNN 게이팅으로 0~9 예측."""
    gate = knn.predict(Xq)               # 0=기존, 1~5=신규(5~9)
    old_pred = base_clf.predict(Xq)      # 동결 분류기 (0~4)
    return np.where(gate == 0, old_pred, gate + 4)


## 5. 평가 — 정확도 · macro-F1 · per-class F1 (BOBAI 방식)


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

val_pred = predict_incremental(X_val)
print(f"Validation accuracy : {accuracy_score(y_val, val_pred):.4f}")
print(f"Validation macro-F1 : {f1_score(y_val, val_pred, average='macro'):.4f}")
print("per-class F1 (0~9)  :", np.round(f1_score(y_val, val_pred, average=None), 3))
print("\n→ 동결 5-way(0~4) 는 base_clf, 신규(5~9) 는 KNN 게이팅으로 처리됨 — 재학습 없이 10-way 달성.")


## 6. 캐글 제출 파일 생성 (`test.csv` → `submission.csv`)


In [ ]:
test_pred = predict_incremental(X_test)
submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame({"ImageId": np.arange(1, len(test_pred) + 1), "Label": test_pred}).to_csv(submission_path, index=False)
print("Saved:", submission_path)
pd.read_csv(submission_path).head()


In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)


## 🏁 제출하기
`submission.csv` 를 [Digit Recognizer](https://www.kaggle.com/competitions/digit-recognizer) 에 제출하세요.

**기법 요약**: 배포된 분류기를 **동결**한 채, 신규 클래스는 **KNN(거리)** 게이팅으로만 추가 — 파라미터 재학습 없이 클래스를 확장하는 *증분/few-shot* 패턴(IOAI Help BOBAI 와 동일).
더 끌어올리려면: KNN 의 `metric='cosine'`·`k` 조정, 참조셋 크기, 또는 동결 임베딩 품질 개선.
